In [14]:
import json
from pathlib import Path
import pandas as pd
from rouge_score import rouge_scorer

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd().parent  

SUMMARY_PATH = BASE_DIR / "data" / "summaries" / "akzhan_sample1_summary.json"
GOLD_PATH = BASE_DIR / "data" / "gold_summaries" / "sample_1_gold.json"


In [16]:
with open(SUMMARY_PATH, "r", encoding="utf-8") as f:
    system_summary = json.load(f)

with open(GOLD_PATH, "r", encoding="utf-8") as f:
    gold = json.load(f)

reference = gold["human_summary"] if isinstance(gold, dict) else gold

extractive = system_summary.get("textrank_summary", "")
abstractive = system_summary.get("abstractive_full_summary", "")

In [17]:
scorer = rouge_scorer.RougeScorer([
    "rouge1",
    "rouge2",
    "rougeL"
], use_stemmer=True)

In [18]:
scores_extractive = scorer.score(reference, extractive)
scores_abstractive = scorer.score(reference, abstractive)

In [19]:
def format_scores(scores):
    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure,
    }

results = pd.DataFrame.from_dict({
    "Extractive (TextRank)": format_scores(scores_extractive),
    "Abstractive (Transformer)": format_scores(scores_abstractive)
}, orient="columns")

results = results.round(4)
results

,Extractive (TextRank),Abstractive (Transformer)
ROUGE-1,0.2938,0.4293
ROUGE-2,0.0566,0.1478
ROUGE-L,0.1312,0.2146


In [20]:
verbose = []

for name, scores in [("Extractive", scores_extractive), ("Abstractive", scores_abstractive)]:
    for k in ["rouge1", "rouge2", "rougeL"]:
        verbose.append({
            "Model": name,
            "Metric": k.upper(),
            "Precision": scores[k].precision,
            "Recall": scores[k].recall,
            "F1": scores[k].fmeasure
        })

pd.DataFrame(verbose).round(4)

,Model,Metric,Precision,Recall,F1
0,Extractive,ROUGE1,0.2597,0.3381,0.2938
1,Extractive,ROUGE2,0.0500,0.0652,0.0566
2,Extractive,ROUGEL,0.1160,0.1511,0.1312
3,Abstractive,ROUGE1,0.6667,0.3165,0.4293
4,Abstractive,ROUGE2,0.2308,0.1087,0.1478
5,Abstractive,ROUGEL,0.3333,0.1583,0.2146
